# Silver — RF-1 Soft-Soil Susceptibility

Clips the bronze-catalogued sources to a **3 km AOI** around Maple Ridge, BC, aligns
everything to a **10 m UTM grid**, computes RF-1 factor metrics, and scores per-pixel
**Susceptibility (S)** and **Consequence (C)** ratings.

| | |
|---|---|
| **Reads (bronze)** | Planetary Computer rasters (Sentinel-1/2, Copernicus DEM, ESA WorldCover, IO LULC, ALOS PALSAR, HGB) · **BC Soil Information Finder Tool** survey polygons (`bronze_bc_soil_survey_polygons`) |
| **Writes (silver)** | `silver_rf1_soil_susceptibility` — one row per 10 m pixel with factor metrics, the soil ground-truth signal, and S/C ratings |
| **Feeds** | `gold_rf1_risk_matrix` → `risk_score = S × C` |

The BC soil survey is **surveyed ground truth** for soft ground (drainage class /
parent material / texture), so it carries the largest weight in Susceptibility and
flows straight through to the gold risk score. Run top-to-bottom with `silver_lakehouse`
attached as the default lakehouse.


## 1. Dependencies

Install the geospatial raster stack used to pull and clip the Planetary Computer
COGs. `requests`/`pyspark` already exist in the Fabric runtime; the libraries
below are the RF-1 analytics stack from the workload spec.

In [ ]:
%pip install -q pystac-client planetary-computer odc-stac odc-geo rioxarray rasterio

## 2. Area of interest — clip to centre point + 3 km buffer

The bronze layer catalogued sources over a 20 km query radius. For the RF-1 silver
layer we **clip to the AOI centre buffered by 3 km** and build a deterministic
10 m analysis grid in UTM 10N (EPSG:32610) so every raster source aligns
pixel-for-pixel.

In [ ]:
import numpy as np
from datetime import datetime, timezone
from pyproj import Transformer
from affine import Affine
from odc.geo.geobox import GeoBox

# --- AOI: Maple Ridge, BC ---
AOI_NAME   = "Maple Ridge BC"
LAT, LON   = 49.2193, -122.5984
BUFFER_KM  = 3.0                     # clip radius around the centre point
RES_M      = 10                      # analysis grid resolution (metres)
UTM_CRS    = "EPSG:32610"            # UTM zone 10N (covers SW British Columbia)
SEASON     = "2024-06-01/2024-09-30" # snow-free growing season for indices/radar

# Project the centre point to UTM and build a deterministic square geobox.
# Built directly from an affine transform (avoids odc-geo from_bbox signature drift).
_to_utm = Transformer.from_crs("EPSG:4326", UTM_CRS, always_xy=True)
_to_wgs = Transformer.from_crs(UTM_CRS, "EPSG:4326", always_xy=True)
cx, cy  = _to_utm.transform(LON, LAT)
half_m  = BUFFER_KM * 1000.0

# snap origin to the RES_M grid so the box is deterministic
minx = float(np.floor((cx - half_m) / RES_M) * RES_M)
maxy = float(np.ceil((cy + half_m) / RES_M) * RES_M)
W    = int(round((2 * half_m) / RES_M))
H    = W
GEOBOX = GeoBox((H, W), Affine(RES_M, 0.0, minx, 0.0, -RES_M, maxy), UTM_CRS)

# WGS84 bbox (minx, miny, maxx, maxy) used for the STAC searches
_ll = _to_wgs.transform(minx, maxy - H * RES_M)
_ur = _to_wgs.transform(minx + W * RES_M, maxy)
BBOX_WGS = [_ll[0], _ll[1], _ur[0], _ur[1]]

print(f"AOI            : {AOI_NAME}  ({LAT}, {LON})")
print(f"Clip buffer    : {BUFFER_KM} km  ->  {2*BUFFER_KM} x {2*BUFFER_KM} km box")
print(f"Analysis grid  : {H} x {W} px @ {RES_M} m  ({UTM_CRS})")
print(f"STAC bbox WGS84: {[round(v, 5) for v in BBOX_WGS]}")
print(f"Season         : {SEASON}")

## 3. Bronze lineage — which catalogued items overlap the AOI

The bronze tables hold STAC **metadata** (item records), not pixels. We read them
from the bronze lakehouse to confirm which catalogued scenes intersect the 3 km
clip, then re-load the actual COG pixels for those collections in the next step.

In [ ]:
from pyspark.sql import functions as F

BRONZE_WS    = "a7d0f907-bf14-4169-8d34-b8765824aa09"
BRONZE_LH    = "fbdd7d1d-00a2-4e0f-84f8-655fce72e4c9"
BRONZE_ABFSS = f"abfss://{BRONZE_WS}@onelake.dfs.fabric.microsoft.com/{BRONZE_LH}/Tables"

bronze_satellite = {
    "sentinel-2-l2a":     "bronze_sentinel_2_l2a",
    "cop-dem-glo-30":     "bronze_cop_dem_glo_30",
    "sentinel-1-rtc":     "bronze_sentinel_1_rtc",
    "esa-worldcover":     "bronze_esa_worldcover",
    "io-lulc-9-class":    "bronze_io_lulc_9_class",
    "alos-palsar-mosaic": "bronze_alos_palsar_mosaic",
    "hgb":                "bronze_hgb",
}

mnx, mny, mxx, mxy = BBOX_WGS
lineage = []
for coll, tbl in bronze_satellite.items():
    try:
        df = spark.read.format("delta").load(f"{BRONZE_ABFSS}/{tbl}")
        overlap = df.filter(
            (F.col("bbox_minx") <= mxx) & (F.col("bbox_maxx") >= mnx) &
            (F.col("bbox_miny") <= mxy) & (F.col("bbox_maxy") >= mny)
        )
        lineage.append((coll, tbl, df.count(), overlap.count()))
    except Exception as e:
        lineage.append((coll, tbl, -1, -1))
        print(f"  (skip {tbl}: {str(e)[:80]})")

print(f"{'collection':<20}{'bronze_items':>14}{'overlap_3km':>14}")
for coll, tbl, total, ov in lineage:
    tot = "n/a" if total < 0 else total
    ovr = "n/a" if ov < 0 else ov
    print(f"{coll:<20}{str(tot):>14}{str(ovr):>14}")

## 4. Extract the clipped pixels from Planetary Computer

Search the same Planetary Computer collections over the 3 km bbox and load the
COG pixels onto the shared `GEOBOX` (10 m, UTM 10N) with `odc.stac.load`. Every
source is reprojected/resampled to the identical grid so they stack cleanly.

In [ ]:
import planetary_computer as pc
import pystac_client
from odc.stac import load as odc_load

CATALOG = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=pc.sign_inplace,
)

def search(collection, bbox, datetime=None, query=None):
    items = list(CATALOG.search(
        collections=[collection], bbox=bbox, datetime=datetime, query=query,
    ).items())
    print(f"  {collection:<20} {len(items):>3} item(s)")
    return items

def load(items, bands, resampling="bilinear"):
    """Load items onto the shared GEOBOX (eager numpy, no dask)."""
    return odc_load(items, bands=bands, geobox=GEOBOX,
                    resampling=resampling, chunks=None)

print("Searching Planetary Computer over the 3 km clip ...")
s2_items  = search("sentinel-2-l2a", BBOX_WGS, SEASON, {"eo:cloud_cover": {"lt": 35}})
dem_items = search("cop-dem-glo-30", BBOX_WGS)
s1_items  = search("sentinel-1-rtc", BBOX_WGS, SEASON)
wc_items  = search("esa-worldcover", BBOX_WGS)

In [ ]:
def _med(ds, dim="time"):
    return ds.median(dim=dim, skipna=True) if dim in ds.dims else ds

# --- Sentinel-2 L2A: surface reflectance (scaled 0-1) -> indices ---
s2 = load(s2_items, ["B02", "B03", "B04", "B08", "B11", "B12"])
s2 = _med(s2) / 10000.0
B02, B03, B04 = s2["B02"], s2["B03"], s2["B04"]
B08, B11, B12 = s2["B08"], s2["B11"], s2["B12"]

# --- Copernicus DEM GLO-30: elevation (m) ---
dem  = load(dem_items, ["data"], resampling="bilinear")
elev = _med(dem)["data"].astype("float32")

# --- Sentinel-1 RTC: linear power -> dB ---
s1 = load(s1_items, ["vv", "vh"])
s1 = _med(s1)
vv_db = 10.0 * np.log10(s1["vv"].where(s1["vv"] > 0))
vh_db = 10.0 * np.log10(s1["vh"].where(s1["vh"] > 0))

# --- ESA WorldCover: categorical land-use (nearest, latest epoch) ---
wc = load(wc_items, ["map"], resampling="nearest")
wc_class = (wc["map"].isel(time=-1) if "time" in wc.dims else wc["map"]).astype("int16")

print("Loaded clipped rasters onto the shared grid:")
for name, da in [("S2 B08", B08), ("DEM", elev), ("S1 vv_db", vv_db), ("WorldCover", wc_class)]:
    print(f"  {name:<12} shape={tuple(da.shape)}  valid={int(np.isfinite(np.asarray(da, dtype='float64')).sum())}")

## 5. RF-1 factor metrics — indices + landform

Per the RF-1 workflow (spec §6) compute the spectral indices (NDVI, NDWI, BSI,
organic-soils ratio), terrain (slope, valley-bottom-flatness proxy) from the
Copernicus DEM, and radar wetness (S1 VV/VH dB). All arrays share the 10 m grid.

In [ ]:
def arr(da):
    return np.asarray(da, dtype="float32")

eps = 1e-6
b2, b3, b4 = arr(B02), arr(B03), arr(B04)
b8, b11, b12 = arr(B08), arr(B11), arr(B12)

# --- spectral indices ---
ndvi = (b8 - b4) / (b8 + b4 + eps)                                  # vegetation
ndwi = (b3 - b8) / (b3 + b8 + eps)                                  # McFeeters water
bsi  = ((b11 + b4) - (b8 + b2)) / ((b11 + b4) + (b8 + b2) + eps)    # bare soil
organic_idx = b11 / (b12 + eps)                                     # SWIR1/SWIR2 organic proxy

# --- terrain from DEM (10 m grid) ---
z = arr(elev)
gy, gx = np.gradient(z, RES_M, RES_M)               # dz/dy, dz/dx in m/m
slope_deg = np.degrees(np.arctan(np.hypot(gx, gy)))

# Valley-bottom-flatness proxy (MrVBF-style): flat + low-lying = high.
def _pct_rank(a):
    f = a[np.isfinite(a)]
    if f.size == 0:
        return np.zeros_like(a)
    order = np.argsort(f)
    ranks = np.empty_like(order, dtype="float32")
    ranks[order] = np.linspace(0.0, 1.0, f.size, dtype="float32")
    out = np.full(a.shape, np.nan, dtype="float32")
    out[np.isfinite(a)] = ranks
    return out

flatness  = np.clip(1.0 - (slope_deg / 12.0), 0.0, 1.0)   # 1 at 0deg, 0 at >=12deg
lowness   = 1.0 - _pct_rank(z)                            # 1 at valley floor
vbf_proxy = np.clip(flatness * lowness, 0.0, 1.0)

# --- radar wetness ---
vv = arr(vv_db)
vh = arr(vh_db)

wcc = np.asarray(wc_class, dtype="int16")

print("Metric ranges (finite):")
for nm, a in [("ndvi", ndvi), ("ndwi", ndwi), ("bsi", bsi), ("organic_idx", organic_idx),
              ("slope_deg", slope_deg), ("vbf_proxy", vbf_proxy), ("vv_db", vv)]:
    f = a[np.isfinite(a)]
    if f.size:
        print(f"  {nm:<12} min={f.min():7.2f}  med={np.median(f):7.2f}  max={f.max():7.2f}")

## 5b. Soil-survey ground truth (BC SIFT)

The satellite/DEM/radar metrics above are *indirect* proxies for soft ground. The
bronze **BC Soil Information Finder Tool** polygons (`bronze_bc_soil_survey_polygons`)
are surveyed ground truth: their drainage class / parent material / texture map
directly onto soft-soil behaviour. We reproject each polygon to the UTM analysis CRS,
score it from those attributes (`organic`/`very poorly drained`/`gley` → high,
`fluvial`/`lacustrine`/`clay` → moderate), and **rasterize that score onto the 10 m
grid** so it can be blended into Susceptibility. Degrades gracefully to an all-zero
signal if the bronze soil table is absent.


In [ ]:
import json
from rasterio.features import rasterize
from rasterio.warp import transform_geom

# --- BC Soil Information Finder Tool (SIFT) polygons from bronze ---
# keyword (drainage / parent material / texture text) -> soft-soil weight (0..1)
SOFT_KEYS = {
    "very poorly": 1.0, "organic": 1.0, "peat": 1.0, "muck": 1.0, "bog": 1.0,
    "gleysol": 0.9, "gley": 0.9, "poorly drained": 0.9, "poorly": 0.85,
    "fluvial": 0.7, "alluv": 0.7, "lacustrine": 0.7, "marine": 0.7, "fen": 0.9,
    "imperfectly": 0.6, "clay": 0.6, "silt": 0.5,
}

def _soft_score(props_json):
    try:
        p = json.loads(props_json) if props_json else {}
    except Exception:
        return 0.4
    blob = " ".join(str(v) for v in p.values()).lower()
    score = 0.0
    for k, w in SOFT_KEYS.items():
        if k in blob:
            score = max(score, w)
    return score if score > 0 else 0.4  # mapped soil, unknown drainage -> mild prior

SOIL_TBL = "bronze_bc_soil_survey_polygons"
shapes = []
try:
    soil_rows = (spark.read.format("delta").load(f"{BRONZE_ABFSS}/{SOIL_TBL}")
                 .select("geometry_json", "properties_json").collect())
    for r in soil_rows:
        try:
            geom = json.loads(r["geometry_json"]) if r["geometry_json"] else None
            if not geom:
                continue
            geom_utm = transform_geom("EPSG:4326", UTM_CRS, geom)
            shapes.append((geom_utm, _soft_score(r["properties_json"])))
        except Exception:
            continue
    print(f"{SOIL_TBL}: {len(soil_rows)} bronze polygons, {len(shapes)} usable geometries")
except Exception as e:
    print(f"  (skip {SOIL_TBL}: {str(e)[:100]})")

aff = GEOBOX.affine
if shapes:
    soil_soft = rasterize(shapes, out_shape=(H, W), transform=aff,
                          fill=0.0, all_touched=True, dtype="float32")
    soil_mapped = rasterize([(g, 1) for g, _ in shapes], out_shape=(H, W),
                            transform=aff, fill=0, all_touched=True,
                            dtype="uint8").astype("float32")
else:
    print("  no soil polygons over the AOI clip -> soil signal is all-zero")
    soil_soft   = np.zeros((H, W), dtype="float32")
    soil_mapped = np.zeros((H, W), dtype="float32")

print(f"Soil-survey ground truth burned onto grid: "
      f"mapped={int(soil_mapped.sum()):,} px, soft>0={int((soil_soft > 0).sum()):,} px")


## 6. Susceptibility (S) and Consequence (C) ratings

- **S (1–5)** — soft-soil susceptibility blended from the **BC soil-survey ground
  truth** (largest weight), the valley-bottom-flatness proxy, radar/optical wetness,
  and bare/organic signals (spec §7). WorldCover wetland classes (90/95) with S1
  VV < −15 dB, and any pixel the survey maps as organic / very-poorly-drained, are
  forced to the top of the scale.
- **C (1–5)** — consequence/exposure proxy from WorldCover land-use (built-up →
  high, water → low), nudged up on steep slopes where movement matters (spec §8).


In [ ]:
def nz(a):
    return np.nan_to_num(a, nan=0.0)

def clip01(a):
    return np.clip(a, 0.0, 1.0)

# --- soft-soil probability P (0..1): availability-weighted blend ---
sig_vbf     = clip01(vbf_proxy)                       # valley-bottom flatness
sig_soil    = clip01(soil_soft)                       # BC soil-survey ground truth
is_wetland  = np.isin(wcc, [90, 95]).astype("float32")
is_water    = (wcc == 80).astype("float32")
wet_radar   = clip01((-(vv) - 12.0) / 8.0)            # VV<-12 -> wet, VV<-20 -> 1
wet_ndwi    = clip01(ndwi * 2.0)
bare        = clip01((bsi + 0.10) / 0.40) * (ndvi < 0.30)
organic     = clip01((organic_idx - 1.0) / 0.5)

# BC SIFT survey is direct evidence, so it carries the single largest weight.
p_soft = clip01(
    0.25 * nz(sig_vbf) +
    0.20 * nz(sig_soil) +
    0.15 * (0.5 * is_wetland + 0.5 * is_water) +
    0.13 * nz(wet_radar) +
    0.12 * nz(wet_ndwi) +
    0.08 * nz(bare) +
    0.07 * nz(organic)
)
# where the survey maps organic / very-poorly-drained ground, lift P to the top
p_soft = np.where(soil_soft >= 0.9, np.maximum(p_soft, 0.85), p_soft)

# --- S rating 1..5 ---
s_rating = np.digitize(p_soft, [0.20, 0.40, 0.60, 0.80]).astype("int16") + 1
s_rating = np.where((is_wetland == 1) & (vv < -15.0), 5, s_rating)
s_rating = np.where((is_wetland == 1) & (s_rating < 4), 4, s_rating)
s_rating = np.where(is_water == 1, 5, s_rating)
s_rating = np.where(soil_soft >= 0.9, np.maximum(s_rating, 4), s_rating)  # surveyed soft soil
s_rating = np.clip(s_rating, 1, 5).astype("int16")

# --- C rating 1..5 from WorldCover exposure, modulated by slope ---
exposure = {10: 2, 20: 3, 30: 3, 40: 4, 50: 5, 60: 2, 70: 1, 80: 1, 90: 2, 95: 2, 100: 1}
c_base = np.vectorize(lambda v: exposure.get(int(v), 2))(wcc).astype("int16")
c_rating = np.where(slope_deg > 15.0, c_base + 1, c_base)
c_rating = np.clip(c_rating, 1, 5).astype("int16")

print("S rating distribution:", {int(k): int(v) for k, v in zip(*np.unique(s_rating, return_counts=True))})
print("C rating distribution:", {int(k): int(v) for k, v in zip(*np.unique(c_rating, return_counts=True))})


## 7. Write the silver table

Flatten the clipped grid to one row per 10 m pixel (with geographic coordinates,
all factor metrics and the S/C ratings) and persist as the Delta table
`silver_rf1_soil_susceptibility` in the silver lakehouse.

In [ ]:
import pandas as pd

# pixel-centre coordinates from the geobox affine
aff = GEOBOX.affine
rows, cols = np.mgrid[0:H, 0:W]
utm_x = aff.c + (cols + 0.5) * aff.a + (rows + 0.5) * aff.b
utm_y = aff.f + (cols + 0.5) * aff.d + (rows + 0.5) * aff.e
lon_g, lat_g = _to_wgs.transform(utm_x, utm_y)

now_utc = datetime.now(timezone.utc).isoformat()

pdf = pd.DataFrame({
    "row":             rows.ravel().astype("int32"),
    "col":             cols.ravel().astype("int32"),
    "utm_x":           utm_x.ravel().astype("float64"),
    "utm_y":           utm_y.ravel().astype("float64"),
    "lon":             lon_g.ravel().astype("float64"),
    "lat":             lat_g.ravel().astype("float64"),
    "elevation_m":     z.ravel().astype("float32"),
    "slope_deg":       slope_deg.ravel().astype("float32"),
    "vbf_proxy":       vbf_proxy.ravel().astype("float32"),
    "ndvi":            ndvi.ravel().astype("float32"),
    "ndwi":            ndwi.ravel().astype("float32"),
    "bsi":             bsi.ravel().astype("float32"),
    "organic_idx":     organic_idx.ravel().astype("float32"),
    "vv_db":           vv.ravel().astype("float32"),
    "vh_db":           vh.ravel().astype("float32"),
    "worldcover_class": wcc.ravel().astype("int16"),
    "soil_soft":       soil_soft.ravel().astype("float32"),
    "soil_mapped":     soil_mapped.ravel().astype("int16"),
    "p_soft":          p_soft.ravel().astype("float32"),
    "s_rating":        s_rating.ravel().astype("int16"),
    "c_rating":        c_rating.ravel().astype("int16"),
})

# keep pixels with valid terrain (inside the DEM footprint)
pdf = pdf[np.isfinite(pdf["elevation_m"])].reset_index(drop=True)
pdf["aoi_name"]       = AOI_NAME
pdf["aoi_lat"]        = LAT
pdf["aoi_lon"]        = LON
pdf["buffer_km"]      = BUFFER_KM
pdf["resolution_m"]   = RES_M
pdf["risk_factor"]    = "RF-1"
pdf["ingested_at_utc"] = now_utc

print(f"Silver pixels: {len(pdf):,}  ({H}x{W} grid, {RES_M} m)")
print(f"  soil-mapped pixels kept: {int(pdf['soil_mapped'].sum()):,}")

sdf = spark.createDataFrame(pdf)
(sdf.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_rf1_soil_susceptibility"))

cnt = spark.read.table("silver_rf1_soil_susceptibility").count()
print(f"Wrote silver_rf1_soil_susceptibility -> {cnt:,} rows")
